# Imports

In [ ]:
!pip install torch_geometric

In [2]:
import os
import glob
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, LayerNorm as GraphLayerNorm

from sklearn.metrics import (precision_score, recall_score, f1_score, fbeta_score,
                             average_precision_score, roc_auc_score, confusion_matrix,
                             precision_recall_curve)


In [4]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Load data

In [5]:
class NF_IDS_Dataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # root_dir: (eg: "./dataset_processed")
        # split: 'train', 'val' or 'test'
        self.split_dir = os.path.join(root_dir, split)
        super().__init__(self.split_dir)

        # List files ordered numerically to respect the time
        self.files = sorted(
            [f for f in os.listdir(self.split_dir) if f.endswith('.pt')],
            key=lambda x: int(x.split('_')[1].split('.')[0])
        )

    def len(self):
        return len(self.files)

    def get(self, idx):
        data = torch.load(
            os.path.join(self.split_dir, self.files[idx]),
            weights_only=False # to allow loading complex graph objects
        )
        return data

# Models

## Simple MLP

In [6]:
class SimpleMLP(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super().__init__()

        # Input: Only flow attributes (edge_attr)
        # The MLP only needs to know how many features the edge has.
        self.input_dim = edge_dim

        # Note: node_dim is received for compatibility but is not used

        # We use LayerNorm instead of BatchNorm1d
        # LayerNorm does not fail with batch_size=1
        self.net = nn.Sequential(
            nn.Linear(self.input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1)
        )

        if output_bias_init is not None:
            self.net[-1].bias.data.fill_(output_bias_init)

    def forward(self, x, edge_index, edge_attr):
        """
        Arguments:
            x, edge_index: These are received to maintain compatibility with
                           train_epoch, but this model ignores them (it doesn't
                           use a graph structure).
            edge_attr: The flow's features (bytes, duration, etc.)
        """
        # We ignore everything except the attributes of the flow
        return self.net(edge_attr)


## Edge GRU

In [7]:
class EdgeGRU_Baseline(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout, output_bias_init=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.node_memory = {}

        # 1. ENCODER
        input_dim = (2 * node_dim) + edge_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        # 2. GRU Cell
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)

        # 3. EDGE CLASSIFIER
        classifier_input_dim = (2 * hidden_dim) + edge_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

    def detach_all_memory(self):
        """
        It cuts off the flow of gradients in stored memory.
        Required for Truncated Backpropagation Through Time (TBPTT).
        """
        for k, v in self.node_memory.items():
            self.node_memory[k] = v.detach()

    def reset_memory(self):
        """Reset memory (useful when starting epoch or validation)"""
        self.node_memory = {}
    # --------------------------------------------------

    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)

        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)

        count[count < 1] = 1
        return out / count


    def forward(self, x, edge_index, edge_attr, global_node_ids):
        device = x.device
        num_nodes_batch = x.size(0)

        src, dst = edge_index

        # Conceptual Difference:
        # ST-GNN: Mixes neighbor information before updating memory.
        # EdgeGRU: Takes DIRECT information from the edge (src+dst+attr) and updates memory.

        raw_features = torch.cat([x[src], x[dst], edge_attr], dim=1)
        encoded_features = self.encoder(raw_features)

        # --- Memory ---
        hidden_states = []
        global_ids_list = global_node_ids.tolist()
        h_prev = torch.zeros(num_nodes_batch, self.hidden_dim, device=device)

        # Restore previous state
        for i, gid in enumerate(global_ids_list):
            if gid in self.node_memory:
                h_prev[i] = self.node_memory[gid]

        # Update GRU (using manual aggregation of incoming edges)
        aggr_input = self.manual_scatter_mean(encoded_features, src, dim_size=num_nodes_batch)
        h_new = self.gru(aggr_input, h_prev)

        # Save new state (detachment is done externally in train_epoch or here if necessary)
        # Note: Normally, we do NOT detach in the forward if we want to train BPTT.
        # Detachment is done in 'detach_all_memory' between batches.
        h_new_stored = h_new.clone()
        for i, gid in enumerate(global_ids_list):
            self.node_memory[gid] = h_new_stored[i]

        # Classify
        h_src = h_new[src]
        h_dst = h_new[dst]
        edge_representation = torch.cat([h_src, h_dst, edge_attr], dim=1)


        return self.classifier(edge_representation)

## Static GNN

In [8]:
class StaticGNN_Identity(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super(StaticGNN_Identity, self).__init__()

        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout

        # IDENTITY PROJECTION
        # Layer to transform the raw edge attributes into a node vector
        self.edge_proj = nn.Linear(edge_dim, node_dim)

        # GNN INPUT DIMENSIONS
        # Since (What the IP sends + What the IP receives) will be concatenated, the input is double.
        gnn_input_dim = 2 * node_dim

        # GNN LAYER 1
        self.gnn1 = GATv2Conv(
            in_channels=gnn_input_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False,
            dropout=dropout
        )
        self.norm1 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # GNN LAYER 2
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False,
            dropout=dropout
        )
        self.norm2 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # EDGE CLASSIFIER
        classifier_input_dim = (2 * hidden_dim) + edge_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Bias Init
        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

    # --- Utils for building "identity" (node features) ---
    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)
        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)
        count[count < 1] = 1
        return out / count

    def forward(self, x, edge_index, edge_attr):
        # Check empty graph
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # ---------------------------------------------------------
        # STEP 1: BUILDING "IDENTITY" (node features)
        # ---------------------------------------------------------
        edge_embeddings = self.edge_proj(edge_attr)
        edge_embeddings = F.relu(edge_embeddings)

        src, dst = edge_index
        num_nodes = x.size(0)

        # Dual Identity
        x_out = self.manual_scatter_mean(edge_embeddings, src, dim_size=num_nodes)
        x_in  = self.manual_scatter_mean(edge_embeddings, dst, dim_size=num_nodes)

        # Concatenate [What IP sends | What IP recibes]
        x_input = torch.cat([x_out, x_in], dim=1)

        # ---------------------------------------------------------
        # STEP 2: GNN LAYERS
        # ---------------------------------------------------------

        # Layer 1
        h1 = self.gnn1(x_input, edge_index, edge_attr=edge_attr)
        h1 = self.norm1(h1) # LayerNorm
        h1 = F.elu(h1)
        h1 = F.dropout(h1, p=self.dropout_rate, training=self.training)

        # Layer 2
        h2 = self.gnn2(h1, edge_index, edge_attr=edge_attr)
        h2 = self.norm2(h2) # LayerNorm
        h2 = F.elu(h2)

        # ---------------------------------------------------------
        # STEP 3: CLASSIFICATION
        # ---------------------------------------------------------
        src, dst = edge_index
        edge_rep = torch.cat([h2[src], h2[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)

## ST-GNN (Ours)

In [9]:
class ST_GNN_Identity(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim, dropout=0.2, output_bias_init=None):
        super(ST_GNN_Identity, self).__init__()

        self.hidden_dim = hidden_dim
        self.dropout_rate = dropout

        # IDENTITY PROJECTION
        # Layer to transform the raw edge attributes into a node vector
        self.edge_proj = nn.Linear(edge_dim, node_dim)

        # GNN INPUT DIMENSIONS
        # Since (What the IP sends + What the IP receives) will be concatenated, the input is double.
        gnn_input_dim = 2 * node_dim

        # GNN LAYER 1
        self.gnn1 = GATv2Conv(
            in_channels=gnn_input_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=2,
            concat=False,
            dropout=dropout
        )
        self.norm1 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # GNN LAYER 2
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            edge_dim=edge_dim,
            heads=1,
            concat=False,
            dropout=dropout
        )
        self.norm2 = GraphLayerNorm(hidden_dim) # to stabilize traffic features

        # TEMPORAL (GRU)
        self.gru = nn.GRUCell(input_size=hidden_dim, hidden_size=hidden_dim)

        # EDGE CLASSIFIER
        classifier_input_dim = (2 * hidden_dim) + edge_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Bias Init
        if output_bias_init is not None:
            self.classifier[-1].bias.data.fill_(output_bias_init)

        self.node_memory = {}

    # --- UTILS ---
    def manual_scatter_mean(self, src, index, dim_size):
        out = torch.zeros((dim_size, src.size(1)), device=src.device)
        out.index_add_(0, index, src)

        ones = torch.ones(src.size(0), 1, device=src.device)
        count = torch.zeros(dim_size, 1, device=src.device)
        count.index_add_(0, index, ones)

        count[count < 1] = 1
        return out / count

    def get_memory(self, ids, device):
        mem_list = []
        for i in ids:
            i = i.item()
            if i in self.node_memory:
                mem_list.append(self.node_memory[i])
            else:
                mem_list.append(torch.zeros(self.hidden_dim, device=device))
        return torch.stack(mem_list)

    def update_memory(self, ids, h_new):
        h_stored = h_new.clone()
        for idx, global_id in enumerate(ids):
            self.node_memory[global_id.item()] = h_stored[idx]

    def detach_all_memory(self):
        for k in self.node_memory:
            self.node_memory[k] = self.node_memory[k].detach()

    def reset_memory(self):
        self.node_memory = {}

    def forward(self, x, edge_index, edge_attr, global_ids):
        if x.size(0) == 0: return torch.empty((0, 1), device=x.device)

        # ---------------------------------------------------------
        # STEP 1: BUILDING "IDENTITY" (node features)
        # ---------------------------------------------------------

        edge_embeddings = self.edge_proj(edge_attr)
        edge_embeddings = F.relu(edge_embeddings) # Activación inicial

        src, dst = edge_index
        num_nodes = x.size(0)

        # Dual Identity
        x_out = self.manual_scatter_mean(edge_embeddings, src, dim_size=num_nodes)
        x_in  = self.manual_scatter_mean(edge_embeddings, dst, dim_size=num_nodes)

        # Concatenate [What IP sends | What IP recibes]
        # Size: [Num_Nodes, 2 * Node_Dim]
        x_input = torch.cat([x_out, x_in], dim=1)

        # ---------------------------------------------------------
        # STEP 2: GNN PROCESSING + MEMORY
        # ---------------------------------------------------------
        h_prev = self.get_memory(global_ids, x.device)

        # GNN LAYER 1
        z = self.gnn1(x_input, edge_index, edge_attr=edge_attr)
        z = self.norm1(z) # LayerNorm
        z = F.elu(z)
        z = F.dropout(z, p=self.dropout_rate, training=self.training)

        # GNN LAYER 2
        z = self.gnn2(z, edge_index, edge_attr=edge_attr)
        z = self.norm2(z)
        z = F.elu(z)

        # GRU LAYER
        h_current = self.gru(z, h_prev)
        self.update_memory(global_ids, h_current)

        # ---------------------------------------------------------
        # STEP 3: CLASSIFICATION
        # ---------------------------------------------------------
        edge_rep = torch.cat([h_current[src], h_current[dst], edge_attr], dim=1)

        return self.classifier(edge_rep)

# Functions

## Metrics

In [10]:
def calculate_metrics_gnn(y_true, y_probs, prob_threshold=0.5):
    """
    y_true: Array of actual labels (0 or 1)
    y_probs: Array de PROBABILITIES (0.0 to 1.0). NO LOGITS.
    """
    y_true = np.array(y_true)
    probs = np.array(y_probs)

    # Predictions (0 or 1) using the threshold
    preds = (probs > prob_threshold).astype(int)

    # Threshold-dependent metrics
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    f2 = fbeta_score(y_true, preds, beta=2, zero_division=0)

    # Threshold-independent metrics
    try:
        ap = average_precision_score(y_true, probs)
        roc = roc_auc_score(y_true, probs)
    except ValueError:
        # This occurs if y_true has only one class (e.g., only benign in the batch)
        ap = 0.0; roc = 0.5

    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "Precision": prec, "Recall": rec, "F1": f1, "F2": f2,
        "AUC-PR": ap, "AUC-ROC": roc, "FPR": fpr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

## Evaluate

In [11]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, is_temporal=False):
    """
    Just run the model and return Loss and raw Probabilities
    """
    model.eval()
    if is_temporal and hasattr(model, 'reset_memory'): model.reset_memory()

    all_probs = []
    all_trues = []
    total_loss = 0
    steps = 0

    # Don't need enumerate here because we're not doing backprop
    for data in loader:
        data = data.to(device)
        if data.x.shape[0] == 0: continue

        if is_temporal:
            out = model(data.x, data.edge_index, data.edge_attr, data.global_node_ids)
        else:
            out = model(data.x, data.edge_index, data.edge_attr)

        # 1. Loss (using logits)
        loss = criterion(out.view(-1), data.y)
        total_loss += loss.item()

        # 2. Probs (using sigmoid)
        probs = torch.sigmoid(out.view(-1))
        all_probs.extend(probs.cpu().numpy())
        all_trues.extend(data.y.cpu().numpy())
        steps += 1

    avg_loss = total_loss / steps if steps > 0 else 0.0

    # Only return the raw data
    return avg_loss, np.array(all_trues), np.array(all_probs)

## Change threshold

In [32]:
def change_threshold(model_class, model_config,
                     val_loader, experiment_name,
                     device='cpu',
                     results_dir="./results_earlystopping"):

    df_metrics = pd.read_csv(f"{results_dir}/logs/{experiment_name}/run_metrics_{experiment_name}.csv")

    for exp_id in range(len(df_metrics)):
        # Setup
        df_exp = df_metrics.iloc[exp_id,:].copy()

        model_config['model_name'] = df_exp['model_name']
        print(f"\nChanging threshold for: {model_config['model_name']}")

        model_config['type'] = df_exp['type']
        run_id = df_exp['extra_run_id']

        pos_weight = torch.tensor([model_config['extra_params']['pos_weight']]).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        is_temporal = 'GRU' in experiment_name or 'ST' in experiment_name

        print("\n OLD METRICS (MAX RECALL FOR PRECISION=0.9):")
        print(f" Precision: {df_exp['Precision']:.4f} | Recall: {df_exp['Recall']:.4f} | F1: {df_exp['F1']:.4f} | F2: {df_exp['F2']:.4f} | AUC-PR: {df_exp['AUC-PR']:.4f} | AUC-ROC: {df_exp['AUC-ROC']:.4f} | FPR: {df_exp['FPR']:.4f}")

        # Charge model
        model_paths = glob.glob(os.path.join(results_dir,"saved_models",experiment_name,f"{run_id}_*.pth"))
        # Ensure we are loading a single file, assuming glob returns a list with one item
        model_path = model_paths[0] if model_paths else None
        if model_path is None:
            print(f"Error: No model found for run_id {run_id} in experiment {experiment_name}")
            continue

        model = model_class(**model_config['model_params']).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))

        # Evaluate
        model.eval()
        _, y_true, y_probs = evaluate(model, val_loader, criterion, device, is_temporal)

        # New threshold (max F1)
        precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
        precisions = precisions[:-1]
        recalls = recalls[:-1]

        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        best_idx = np.argmax(f1_scores)
        new_best_th = thresholds[best_idx]

        # Calculate metrics with new threshold
        new_metrics = calculate_metrics_gnn(y_true, y_probs, prob_threshold=new_best_th)
        new_metrics['optimal_threshold'] = new_best_th

        print("\n NEW METRICS (MAX F1):")
        print(f" Precision: {new_metrics['Precision']:.4f} | Recall: {new_metrics['Recall']:.4f} | F1: {new_metrics['F1']:.4f} | F2: {new_metrics['F2']:.4f} | AUC-PR: {new_metrics['AUC-PR']:.4f} | AUC-ROC: {new_metrics['AUC-ROC']:.4f} | FPR: {new_metrics['FPR']:.4f}\n")

        # Save
        df_new_row = pd.DataFrame([new_metrics]) # Create a DataFrame for the new row
        # Select relevant columns from df_exp for metadata, and then combine with new_metrics
        metadata_for_new_row = df_exp[['run_id', 'model_name', 'type']].to_dict()
        for key, value in metadata_for_new_row.items():
            df_new_row[key] = value

        filepath = f"{results_dir}/logs/{experiment_name}/metrics_newth_{experiment_name}.csv"
        if os.path.exists(filepath):
            df_new_row.to_csv(filepath, mode="a", header=False, index=False)
        else:
            df_new_row.to_csv(filepath, mode="w", header=True, index=False)

        print("-" * 60)

    # Re-reading to get the full updated CSV content if it was saved
    if os.path.exists(filepath):
        return pd.read_csv(filepath)
    else:
        return pd.DataFrame([new_metrics]) # Fallback if file was never created (e.g., error in glob)

# Configuration

In [13]:
ROOT_PATH = "./dataset_processed"

In [37]:
# Instantiate Dataset (Only reads file names)
train_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='train')
val_dataset = NF_IDS_Dataset(root_dir=ROOT_PATH, split='val')

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

# Instantiate DataLoader (Load manager)
# batch_size=1 : Important for ST-GNN to handle memory step by step
# num_workers=2 : Use 2 CPU cores to load files while training
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False) # pin_memory=False for CPU
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, persistent_workers=True, pin_memory=False)

Train size: 1998 | Val size: 428


In [15]:
# --- PARAMETERS ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

EPOCHS = 60
BATCH_STEPS = 10 # backprop every 10 snapshots (sequence)
LR = 0.005
POS_WEIGHT = 2.0

NODE_DIM = 16   # Features dummy (1s)
EDGE_DIM = 32   # 20 numeric + 7 dst_port + 5 protocol
HIDDEN_DIM = 32
DROPOUT = 0.2
BIAS_VALUE = -2.9968

#PROB_THRESHOLD = 0.5



Using device: cpu


In [38]:
model_config = {
    "model_name": None,
    "type": None,
    "model_params": {
        "node_dim": NODE_DIM,
        "edge_dim": EDGE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "output_bias_init": BIAS_VALUE
    },
    #"prob_threshold": PROB_THRESHOLD,
    "extra_params": {
        "epochs": EPOCHS,
        "batch_steps": BATCH_STEPS,
        "pos_weight": POS_WEIGHT,
        "learning_rate": LR
    }
}

# Main

In [30]:
RESULTS_DIR = "./results_earlystopping"

pairs = [(SimpleMLP, "SimpleMLP_BiasOn"),
          (EdgeGRU_Baseline, "EdgeGRU_BiasOn"),
          (StaticGNN_Identity, "StaticGNN_BiasOn_robust_Identity"),
          (ST_GNN_Identity, "ST_GNN_BiasOn_robust_Identity_clone")]

In [39]:
for (mclass, exp_name) in pairs:
    _=change_threshold(mclass, model_config, val_loader, exp_name, DEVICE, RESULTS_DIR)
    print("="*70)
    print("\n")
    print("="*70)


Changing threshold for: SimpleMLP_BiasOn_seed42

 OLD METRICS (MAX RECALL FOR PRECISION=0.9):
 Precision: 0.9015 | Recall: 0.0190 | F1: 0.0372 | F2: 0.0236 | AUC-PR: 0.1836 | AUC-ROC: 0.7786 | FPR: 0.0001

 NEW METRICS (MAX F1):
 Precision: 0.4725 | Recall: 0.1531 | F1: 0.2313 | F2: 0.1771 | AUC-PR: 0.1836 | AUC-ROC: 0.7786 | FPR: 0.0071

------------------------------------------------------------

Changing threshold for: SimpleMLP_BiasOn_seed123

 OLD METRICS (MAX RECALL FOR PRECISION=0.9):
 Precision: 0.4528 | Recall: 0.1565 | F1: 0.2326 | F2: 0.1800 | AUC-PR: 0.1736 | AUC-ROC: 0.7757 | FPR: 0.0078

 NEW METRICS (MAX F1):
 Precision: 0.4528 | Recall: 0.1565 | F1: 0.2326 | F2: 0.1800 | AUC-PR: 0.1736 | AUC-ROC: 0.7757 | FPR: 0.0078

------------------------------------------------------------

Changing threshold for: SimpleMLP_BiasOn_seed777

 OLD METRICS (MAX RECALL FOR PRECISION=0.9):
 Precision: 0.9142 | Recall: 0.0435 | F1: 0.0831 | F2: 0.0537 | AUC-PR: 0.2251 | AUC-ROC: 0.8106 